# Experiment: SciSciNet 500k Tinker RL Smoke

Objective:
- Load the `disruption_novelty_sciscinet_500k` dataset files directly from `~/Downloads`.
- Reuse the repository Tinker RL helpers (`SciSciNetRLDataset`, prompt adapter/renderer, leakage checks).
- Run a tiny offline Tinker-style rollout in-notebook before (optionally) launching the dry-run trainer script.


In [1]:
from __future__ import annotations

import asyncio
import json
import random
import statistics
import sys
from pathlib import Path
from typing import Any

SEED = 20260224
random.seed(SEED)

def find_tinker_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "training" / "tinker_rl" / "tinker_dataset.py").exists() and (candidate / "tinker_disruption_rl").exists():
            return candidate
        nested = candidate / "tinker"
        if (nested / "training" / "tinker_rl" / "tinker_dataset.py").exists() and (nested / "tinker_disruption_rl").exists():
            return nested
    raise FileNotFoundError("Could not locate the tinker repo root from the current working directory.")

TINKER_ROOT = find_tinker_root()
if str(TINKER_ROOT) not in sys.path:
    sys.path.insert(0, str(TINKER_ROOT))

from training.rlvr.metrics import load_json
from training.tinker_rl.tinker_dataset import SciSciNetRLDataset, build_batch_histogram_summary
from training.tinker_rl.tinker_env_adapter import (
    Action,
    DISRUPTION_LABELS,
    SimpleMessageRenderer,
    find_prompt_leakage_markers,
)

print("TINKER_ROOT:", TINKER_ROOT)
print("SEED:", SEED)

TINKER_ROOT: /Users/akhilpandey/code/writing/tinker
SEED: 20260224


## Dataset Paths

This notebook expects the 500k SciSciNet exports to be in `~/Downloads` with the exact filenames you listed.
If you moved them elsewhere, override the `Path(...)` assignments in the next cell.


In [2]:
DOWNLOADS = Path.home() / "Downloads"
DATASET_JSONL = DOWNLOADS / "disruption_novelty_sciscinet_500k.jsonl"
METADATA_JSON = DOWNLOADS / "disruption_novelty_sciscinet_500k.metadata.json"
SPLITS_JSON = DOWNLOADS / "disruption_novelty_sciscinet_500k.splits.json"

for path in (DATASET_JSONL, METADATA_JSON, SPLITS_JSON):
    print(f"{path} :: exists={path.exists()}")

missing = [str(path) for path in (DATASET_JSONL, METADATA_JSON, SPLITS_JSON) if not path.exists()]
if missing:
    raise FileNotFoundError("Missing expected dataset files:\\n- " + "\\n- ".join(missing))

metadata = load_json(METADATA_JSON)
splits = load_json(SPLITS_JSON)

def first_jsonl_record(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        for line_no, raw in enumerate(handle, start=1):
            line = raw.strip()
            if not line:
                continue
            try:
                return json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL at {path}:{line_no}: {exc}") from exc
    raise ValueError(f"No JSON records found in {path}")

sample_record = first_jsonl_record(DATASET_JSONL)
dataset_overview = {
    "metadata_keys": sorted(metadata.keys()),
    "split_counts": splits.get("counts"),
    "split_names": sorted((splits.get("ids") or {}).keys()),
    "sample_openalex_id": sample_record.get("openalex_id"),
    "sample_title": str(sample_record.get("title", ""))[:160],
    "sample_disruption_label": sample_record.get("disruption_label"),
}
dataset_overview

/Users/akhilpandey/Downloads/disruption_novelty_sciscinet_500k.jsonl :: exists=True
/Users/akhilpandey/Downloads/disruption_novelty_sciscinet_500k.metadata.json :: exists=True
/Users/akhilpandey/Downloads/disruption_novelty_sciscinet_500k.splits.json :: exists=True


{'metadata_keys': ['artifacts',
  'generated_at_utc',
  'label_counts',
  'label_thresholds',
  'mode',
  'query_filters',
  'record_count',
  'schema_fields',
  'split_counts',
  'split_ids_sha256',
  'split_ratios',
  'split_seed'],
 'split_counts': {'test': 49182, 'train': 393456, 'val': 49182},
 'split_names': ['test', 'train', 'val'],
 'sample_openalex_id': 'W2889060997',
 'sample_title': 'Valdemar’s Abjection: Poe, Kristeva, Masculinity, and Victim-Monsters',
 'sample_disruption_label': 'neutral'}

## Build The Repository RL Dataset Wrapper

This uses the same helper classes as `training/tinker_rl/train_tinker_rl.py`, but with a notebook-sized subset and local `SimpleMessageRenderer`.


In [3]:
renderer = SimpleMessageRenderer()
SYSTEM_PROMPT = (
    "You are a scientific impact prediction model. Predict the disruption label and provide brief reasoning. "
    "Follow the output format exactly."
)

experiment_cfg = {
    "group_size": 2,
    "batch_size": 6,
    "max_env_tokens": 128,
    "prompt_max_chars": 1800,
    "sampling_strategy": "stratified",
    "seed": SEED,
    "train_split": "train",
    "val_split": "val",
    "max_train": 24,
    "max_val": 6,
    "include_concepts": False,
}

dataset, dataset_info = SciSciNetRLDataset.from_files(
    dataset_jsonl=DATASET_JSONL,
    splits_json=SPLITS_JSON,
    renderer=renderer,
    group_size=experiment_cfg["group_size"],
    batch_size=experiment_cfg["batch_size"],
    system_prompt=SYSTEM_PROMPT,
    max_env_tokens=experiment_cfg["max_env_tokens"],
    prompt_max_chars=experiment_cfg["prompt_max_chars"],
    sampling_strategy=experiment_cfg["sampling_strategy"],
    seed=experiment_cfg["seed"],
    train_split=experiment_cfg["train_split"],
    val_split=experiment_cfg["val_split"],
    max_train=experiment_cfg["max_train"],
    max_val=experiment_cfg["max_val"],
    include_concepts=experiment_cfg["include_concepts"],
)

batch_histograms = []
for i in range(3):
    builders_i = dataset.get_batch(i)
    batch_histograms.append(dataset.observed_batch_label_histogram(builders_i))
sampling_preview = build_batch_histogram_summary(batch_histograms)

{
    "dataset_info": dataset_info,
    "sampling_manifest": dataset.sampling_manifest(),
    "sampling_preview_first_3_batches": sampling_preview,
}

{'dataset_info': {'dataset_jsonl': '/Users/akhilpandey/Downloads/disruption_novelty_sciscinet_500k.jsonl',
  'splits_json': '/Users/akhilpandey/Downloads/disruption_novelty_sciscinet_500k.splits.json',
  'train_split': 'train',
  'val_split': 'val',
  'requested_train_ids': 24,
  'requested_val_ids': 6,
  'loaded_train_records': 24,
  'loaded_val_records': 6,
  'train_limit_mode': 'stratified_scan',
  'split_counts': {'test': 49182, 'train': 393456, 'val': 49182},
  'split_seed': 20260220,
  'train_label_histogram': {'disruptive': 8, 'consolidating': 8, 'neutral': 8},
  'val_label_histogram': {'disruptive': 0, 'consolidating': 0, 'neutral': 6}},
 'sampling_manifest': {'sampling_strategy': 'stratified',
  'seed': 20260224,
  'label_order_cycle': ['disruptive', 'consolidating', 'neutral'],
  'group_size': 2,
  'batch_size': 6,
  'train_label_histogram_natural': {'disruptive': 8,
   'consolidating': 8,
   'neutral': 8},
  'val_label_histogram_natural': {'disruptive': 0,
   'consolidating'

## Prompt Preview + Leakage Check

The adapter intentionally removes target metrics (`CD Index`, novelty scores, etc.) from the prompt.


In [4]:
preview_prompt = dataset.preview_train_prompt()
leak_markers = find_prompt_leakage_markers(preview_prompt)

print(preview_prompt[:1500] + ("..." if len(preview_prompt) > 1500 else ""))
{"prompt_length_chars": len(preview_prompt), "leak_markers": leak_markers}

Predict the disruption label for the paper.
Allowed labels: disruptive, consolidating, neutral.
Return exactly:
disruption: <label>
reasoning: <short justification>

Title: Abstracts
Abstract: Clinicians are putting increasing faith in medical imaging to detect and diagnose disease, and this is contributing to ever increasing healthcare expenditure.Given the long history of ultrasound and its clear benefits over many other modalities, why do clinicians view it with suspicion?Why are radiologists and sonographers prepared to sacrifice this vital, accurate and cost effective modality?Can this decline be reversed, or is it too late?SESSION 2 1.00-3.00pm ABDOMINAL What is an excellent abdominal ultrasound?F Temple St Vincent's Hospital Melbourne Background: An abdominal ultrasound is the sonographer's bread and butter but, when performed excellently, it is also very challenging.What differentiates an excellent abdominal ultrasound from an ordinary one?And what are the consequences of an or

{'prompt_length_chars': 1570, 'leak_markers': []}

## Tiny Offline Tinker-Style Rollout

This runs the repository `TinkerDisruptionEnv` adapter via `DisruptionEnvGroupBuilder.make_envs()` and scores a few toy policies without any hosted Tinker service.


In [ ]:
POLICY_SEED_OFFSETS = {"oracle": 1, "always_neutral": 2, "random": 3}

def make_action_from_text(text: str):
    tokens = renderer.encode_assistant_content(text)
    try:
        return Action(tokens=tokens)
    except Exception:
        return tokens

def choose_label(policy: str, gold_label: str, rng: random.Random) -> str:
    if policy == "oracle":
        return gold_label
    if policy == "always_neutral":
        return "neutral"
    if policy == "random":
        return rng.choice(list(DISRUPTION_LABELS))
    raise ValueError(f"Unsupported policy: {policy}")

def format_response(pred_label: str, policy: str, env_index: int) -> str:
    return (
        f"disruption: {pred_label}\n"
        f"reasoning: {policy} smoke policy prediction (env={env_index})."
    )

async def evaluate_policy(builders, policy: str) -> dict[str, Any]:
    rng = random.Random(SEED + POLICY_SEED_OFFSETS[policy])
    rows = []
    for builder_index, builder in enumerate(builders):
        envs = await builder.make_envs()
        group_rewards = []
        group_predictions = []
        prompt_tokens = None
        for env_index, env in enumerate(envs):
            obs, _stop = await env.initial_observation()
            if prompt_tokens is None:
                prompt_tokens = len(getattr(obs, "tokens", []))
            pred = choose_label(policy, builder.disruption_label, rng)
            response_text = format_response(pred, policy=policy, env_index=env_index)
            result = await env.step(make_action_from_text(response_text))
            group_predictions.append(pred)
            group_rewards.append(float(result.reward))
        rows.append(
            {
                "builder_index": builder_index,
                "openalex_id": builder.openalex_id,
                "gold_label": builder.disruption_label,
                "predictions": group_predictions,
                "group_rewards": group_rewards,
                "group_mean_reward": statistics.fmean(group_rewards),
                "prompt_tokens": prompt_tokens,
            }
        )
    return {"policy": policy, "rows": rows}

def summarize_policy_run(run: dict[str, Any]) -> dict[str, Any]:
    rows = run["rows"]
    flat_rewards = [reward for row in rows for reward in row["group_rewards"]]
    total_preds = sum(len(row["predictions"]) for row in rows)
    exact_matches = sum(
        1
        for row in rows
        for pred in row["predictions"]
        if pred == row["gold_label"]
    )
    return {
        "policy": run["policy"],
        "n_items": len(rows),
        "n_predictions": total_preds,
        "accuracy": (exact_matches / total_preds) if total_preds else 0.0,
        "mean_reward": statistics.fmean(flat_rewards) if flat_rewards else 0.0,
        "min_reward": min(flat_rewards) if flat_rewards else 0.0,
        "max_reward": max(flat_rewards) if flat_rewards else 0.0,
        "avg_prompt_tokens": statistics.fmean([row["prompt_tokens"] for row in rows]) if rows else 0.0,
    }

builders = dataset.get_batch(10_000)
batch_histogram = dataset.observed_batch_label_histogram(builders)

oracle_run = await evaluate_policy(builders, "oracle")
neutral_run = await evaluate_policy(builders, "always_neutral")
random_run = await evaluate_policy(builders, "random")

comparison = [
    summarize_policy_run(oracle_run),
    summarize_policy_run(neutral_run),
    summarize_policy_run(random_run),
]

{
    "batch_histogram": batch_histogram,
    "comparison": comparison,
    "oracle_examples": oracle_run["rows"][:2],
}


## Optional: Launch The Repository Dry-Run Trainer (Same Helpers, End-to-End)

Set `RUN_TRAIN_DRY_RUN = True` to execute `training/tinker_rl/train_tinker_rl.py` on a tiny subset using your `~/Downloads` files.
This remains local/offline (`--dry-run`) and writes artifacts under `tinker/results/tinker_rl/`.


In [ ]:
import shlex
import subprocess

RUN_TRAIN_DRY_RUN = False

train_cmd = [
    sys.executable,
    str(TINKER_ROOT / "training" / "tinker_rl" / "train_tinker_rl.py"),
    "--dataset-jsonl", str(DATASET_JSONL),
    "--splits-json", str(SPLITS_JSON),
    "--output-dir", str(TINKER_ROOT / "results" / "tinker_rl"),
    "--run-name", "notebook_sciscinet_500k_smoke",
    "--dry-run",
    "--overwrite",
    "--epochs", "1",
    "--group-size", "2",
    "--batch-size", "6",
    "--max-train", "24",
    "--max-val", "6",
    "--sampling-strategy", "stratified",
    "--prompt-max-chars", "1800",
]

print(shlex.join([str(part) for part in train_cmd]))

if RUN_TRAIN_DRY_RUN:
    completed = subprocess.run(
        [str(part) for part in train_cmd],
        cwd=str(TINKER_ROOT),
        check=True,
        text=True,
    )
    print("returncode:", completed.returncode)


## Pure Tinker / Cookbook Hosted RL Path (Notebook-Local Env)

This section follows the installed Cookbook patterns from `recipes/rl_basic.py` and `recipes/rl_loop.py`, but defines the SciSciNet disruption environment, group builder, and dataset builder directly in the notebook.

Important: run this notebook with the `mlx` Python kernel (`/Users/akhilpandey/mlx/bin/python`) and set `TINKER_API_KEY` before enabling the final training cell.


In [1]:
import importlib.util
import json
import os
import random
import re
import shutil
import sys
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Mapping, Sequence

if "SEED" not in globals():
    SEED = 20260224

if "TINKER_ROOT" not in globals():
    def _find_tinker_root_for_pure(start: Path | None = None) -> Path:
        start = (start or Path.cwd()).resolve()
        for candidate in (start, *start.parents):
            if (candidate / "tinker_disruption_rl").exists() and (candidate / "training").exists():
                return candidate
            nested = candidate / "tinker"
            if (nested / "tinker_disruption_rl").exists() and (nested / "training").exists():
                return nested
        raise FileNotFoundError("Could not locate tinker repo root for pure-Cookbook path")
    TINKER_ROOT = _find_tinker_root_for_pure()

ENV_FILE = TINKER_ROOT / ".env"
loaded_env_keys: list[str] = []
if ENV_FILE.exists():
    try:
        from dotenv import load_dotenv  # type: ignore

        load_dotenv(ENV_FILE, override=False)
        loaded_env_keys.append("python-dotenv")
    except Exception:
        # Minimal .env parser fallback (KEY=VALUE, optional quotes, ignores comments).
        for raw_line in ENV_FILE.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[len("export ") :].strip()
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip()
            if not key:
                continue
            if value and len(value) >= 2 and value[0] == value[-1] and value[0] in {'"', "'"}:
                value = value[1:-1]
            os.environ.setdefault(key, value)
        loaded_env_keys.append("manual-parser")

if "DATASET_JSONL" not in globals() or "SPLITS_JSON" not in globals():
    _downloads = Path.home() / "Downloads"
    DATASET_JSONL = _downloads / "disruption_novelty_sciscinet_500k.jsonl"
    SPLITS_JSON = _downloads / "disruption_novelty_sciscinet_500k.splits.json"

print("Notebook kernel:", sys.executable)
print("Expected mlx kernel:", "/Users/akhilpandey/mlx/bin/python")
print("TINKER_ROOT:", TINKER_ROOT)
print("SEED:", SEED)
print(".env file:", ENV_FILE, "exists=", ENV_FILE.exists(), "loader=", loaded_env_keys or None)
print("tinker importable:", bool(importlib.util.find_spec("tinker")))
print("tinker_cookbook importable:", bool(importlib.util.find_spec("tinker_cookbook")))
print("TINKER_API_KEY set:", bool(os.environ.get("TINKER_API_KEY")))
print("TINKER_BASE_URL:", os.environ.get("TINKER_BASE_URL"))

if not importlib.util.find_spec("tinker") or not importlib.util.find_spec("tinker_cookbook"):
    raise ImportError(
        "This section requires tinker + tinker_cookbook in the active notebook kernel. "
        "Use the mlx env or install both packages in your current kernel."
    )

import tinker
from tinker_cookbook import model_info, renderers
from tinker_cookbook.rl import train as cookbook_rl_train
from tinker_cookbook.rl.types import Env as CookbookEnv
from tinker_cookbook.rl.types import EnvGroupBuilder as CookbookEnvGroupBuilder
from tinker_cookbook.rl.types import RLDataset as CookbookRLDataset
from tinker_cookbook.rl.types import StepResult as CookbookStepResult
from tinker_cookbook.rl.types import Trajectory
from tinker_cookbook.tokenizer_utils import get_tokenizer
from tinker_cookbook.utils import logtree

Notebook kernel: /Users/akhilpandey/mlx/bin/python3
Expected mlx kernel: /Users/akhilpandey/mlx/bin/python
TINKER_ROOT: /Users/akhilpandey/code/writing/tinker
SEED: 20260224
.env file: /Users/akhilpandey/code/writing/tinker/.env exists= True loader= ['python-dotenv']
tinker importable: True
tinker_cookbook importable: True
TINKER_API_KEY set: True
TINKER_BASE_URL: None


In [2]:
# Notebook-local pure Tinker/Cookbook env + dataset implementation

DISRUPTION_LABELS_PURE = ("disruptive", "consolidating", "neutral")
FORBIDDEN_PROMPT_MARKERS_PURE = (
    "CD Index:",
    "Novelty Score:",
    "Conventionality Score:",
    "cd_index",
    "novelty_score",
    "conventionality_score",
)

def _norm_text(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()

def _truncate_with_ellipsis(text: str, max_chars: int) -> str:
    if max_chars <= 0:
        return ""
    if len(text) <= max_chars:
        return text
    if max_chars <= 3:
        return text[:max_chars]
    return text[: max_chars - 3].rstrip() + "..."

def _truncate_title_abstract(title: str, abstract: str, max_total_chars: int) -> tuple[str, str]:
    title_clean = _norm_text(title)
    abstract_clean = _norm_text(abstract)
    if max_total_chars <= 0:
        return "", ""
    if len(title_clean) + len(abstract_clean) <= max_total_chars:
        return title_clean, abstract_clean
    if len(title_clean) >= max_total_chars:
        return _truncate_with_ellipsis(title_clean, max_total_chars), ""
    abstract_budget = max_total_chars - len(title_clean)
    return title_clean, _truncate_with_ellipsis(abstract_clean, abstract_budget)

def _format_concepts(concepts: Any) -> str | None:
    if concepts is None:
        return None
    if isinstance(concepts, (str, bytes, bytearray)):
        raw_items = [concepts]
    elif isinstance(concepts, Sequence):
        raw_items = list(concepts)
    else:
        raw_items = [concepts]
    values: list[str] = []
    seen: set[str] = set()
    for item in raw_items:
        value = _norm_text(item)
        if not value:
            continue
        key = value.lower()
        if key in seen:
            continue
        seen.add(key)
        values.append(value)
    return ", ".join(values[:8]) if values else None

def build_pure_disruption_prompt(record: Mapping[str, Any], *, prompt_max_chars: int, include_concepts: bool) -> str:
    title, abstract = _truncate_title_abstract(
        str(record.get("title", "")),
        str(record.get("abstract", "")),
        int(prompt_max_chars),
    )
    field = _norm_text(record.get("primary_field", "Unknown")) or "Unknown"
    lines = [
        "Predict the disruption label for the paper.",
        "Allowed labels: disruptive, consolidating, neutral.",
        "Return exactly:",
        "disruption: <label>",
        "reasoning: <short justification>",
        "",
        f"Title: {title}",
        f"Abstract: {abstract}",
        f"Year: {int(record.get("publication_year", 0))}",
        f"Citations: {int(record.get("cited_by_count", 0))}",
        f"Field: {field}",
    ]
    if include_concepts:
        concepts_text = _format_concepts(record.get("concepts"))
        if concepts_text:
            lines.append(f"Concepts: {concepts_text}")
    prompt = "\n".join(lines)
    leakage = [m for m in FORBIDDEN_PROMPT_MARKERS_PURE if m in prompt]
    if leakage:
        raise ValueError(f"Prompt leakage markers detected: {leakage}")
    return prompt

def _extract_disruption_label_and_reasoning(text: str) -> tuple[str | None, str]:
    label_match = re.search(
        r"(?im)^\s*disruption\s*:\s*(disruptive|consolidating|neutral)\s*$",
        text or "",
    )
    reasoning_match = re.search(r"(?im)^\s*reasoning\s*:\s*(.+)$", text or "")
    label = label_match.group(1).lower() if label_match else None
    reasoning = _norm_text(reasoning_match.group(1)) if reasoning_match else ""
    return label, reasoning

def _reasoning_bonus(reasoning: str) -> float:
    n = len(_norm_text(reasoning))
    if n == 0:
        return 0.0
    if n < 20:
        return 0.1
    if n < 60:
        return 0.3
    return 0.5

def _normalize_label(value: Any) -> str:
    return str(value or "").strip().lower()

def label_histogram_pure(records: Sequence[Mapping[str, Any]]) -> dict[str, int]:
    counts = Counter(_normalize_label(r.get("disruption_label")) for r in records)
    return {label: int(counts.get(label, 0)) for label in DISRUPTION_LABELS_PURE}

def load_split_records_streaming_head_pure(
    *,
    dataset_jsonl: Path,
    splits_json: Path,
    train_split: str = "train",
    val_split: str = "val",
    max_train: int | None = None,
    max_val: int | None = None,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:
    splits_payload = json.loads(Path(splits_json).read_text(encoding="utf-8"))
    ids_by_split = dict((splits_payload.get("ids") or {}))
    train_ids = [str(x) for x in ids_by_split.get(train_split, [])]
    val_ids = [str(x) for x in ids_by_split.get(val_split, [])]
    if max_train is not None:
        train_ids = train_ids[: max(0, int(max_train))]
    if max_val is not None:
        val_ids = val_ids[: max(0, int(max_val))]
    wanted = set(train_ids) | set(val_ids)
    by_id: dict[str, dict[str, Any]] = {}
    with Path(dataset_jsonl).open("r", encoding="utf-8") as handle:
        for line_no, raw in enumerate(handle, start=1):
            line = raw.strip()
            if not line:
                continue
            record = json.loads(line)
            rid = str(record.get("openalex_id"))
            if rid not in wanted:
                continue
            if rid not in by_id:
                by_id[rid] = dict(record)
            if len(by_id) == len(wanted):
                break
    train_records = [by_id[rid] for rid in train_ids if rid in by_id]
    val_records = [by_id[rid] for rid in val_ids if rid in by_id]
    info = {
        "dataset_jsonl": str(dataset_jsonl),
        "splits_json": str(splits_json),
        "train_split": train_split,
        "val_split": val_split,
        "requested_train_ids": len(train_ids),
        "requested_val_ids": len(val_ids),
        "loaded_train_records": len(train_records),
        "loaded_val_records": len(val_records),
        "train_label_histogram": label_histogram_pure(train_records),
        "val_label_histogram": label_histogram_pure(val_records),
    }
    return train_records, val_records, info

class SciSciNetDisruptionEnvCookbook(CookbookEnv):
    def __init__(
        self,
        *,
        record: Mapping[str, Any],
        renderer: Any,
        system_prompt: str | None,
        max_tokens: int,
        prompt_max_chars: int,
        include_concepts: bool = False,
    ) -> None:
        self.record = dict(record)
        self.renderer = renderer
        self.system_prompt = _norm_text(system_prompt) if system_prompt else None
        self.max_tokens = int(max_tokens)
        self.prompt_max_chars = int(prompt_max_chars)
        self.include_concepts = bool(include_concepts)
        self._done = False
        self._last_prompt_text = ""
        self._last_response_text = ""

    def _stop_condition(self):
        # Cookbook renderer stop sequences (plus hard token cap from rl.train.Config.max_tokens).
        return self.renderer.get_stop_sequences()

    def _prompt_text(self) -> str:
        return build_pure_disruption_prompt(
            self.record,
            prompt_max_chars=self.prompt_max_chars,
            include_concepts=self.include_concepts,
        )

    async def initial_observation(self):
        prompt_text = self._prompt_text()
        self._last_prompt_text = prompt_text
        messages: list[dict[str, Any]] = []
        if self.system_prompt:
            messages.append({"role": "system", "content": self.system_prompt})
        messages.append({"role": "user", "content": prompt_text})
        return self.renderer.build_generation_prompt(messages), self._stop_condition()

    async def step(self, action):
        if self._done:
            raise RuntimeError("Environment already finished")
        self._done = True

        message, parse_success = self.renderer.parse_response(action)
        try:
            content = renderers.get_text_content(message)
        except Exception:
            content = str(getattr(message, "content", "")) if not isinstance(message, dict) else str(message.get("content", ""))
        self._last_response_text = content

        pred_label, reasoning = _extract_disruption_label_and_reasoning(content)
        gold_label = _normalize_label(self.record.get("disruption_label"))
        correctness = 1.0 if pred_label == gold_label and pred_label is not None else -1.0
        r_reasoning = _reasoning_bonus(reasoning)
        total_reward = correctness + r_reasoning

        logtree.log_text(f"OpenAlex ID: {self.record.get("openalex_id")}")
        logtree.log_text(f"Gold disruption: {gold_label}")
        logtree.log_text(f"Predicted disruption: {pred_label}")
        logtree.log_text(f"Response preview: {content[:500]}")
        logtree.log_text(f"Reward: {total_reward:.3f} (correct={correctness:.3f}, reasoning={r_reasoning:.3f})")

        return CookbookStepResult(
            reward=float(total_reward),
            episode_done=True,
            next_observation=tinker.ModelInput.empty(),
            next_stop_condition=self._stop_condition(),
            metrics={
                "R_correctness": float(correctness),
                "R_reasoning": float(r_reasoning),
                "parse_success": int(bool(parse_success)),
                "label_parsed": int(pred_label is not None),
                "label_correct": int(pred_label == gold_label if pred_label is not None else 0),
            },
            logs={
                "openalex_id": str(self.record.get("openalex_id", "")),
                "gold_disruption": gold_label,
                "pred_disruption": str(pred_label or ""),
                "prompt_chars": len(self._last_prompt_text),
                "response_chars": len(content),
            },
        )

@dataclass(frozen=True)
class SciSciNetEnvGroupBuilderCookbook(CookbookEnvGroupBuilder):
    record: Mapping[str, Any]
    group_size: int
    renderer: Any
    system_prompt: str | None
    max_tokens: int
    prompt_max_chars: int
    include_concepts: bool = False

    @property
    def openalex_id(self) -> str:
        return str(self.record.get("openalex_id", ""))

    @property
    def disruption_label(self) -> str:
        return _normalize_label(self.record.get("disruption_label"))

    def build_prompt_text(self) -> str:
        return build_pure_disruption_prompt(
            self.record,
            prompt_max_chars=self.prompt_max_chars,
            include_concepts=self.include_concepts,
        )

    async def make_envs(self) -> Sequence[CookbookEnv]:
        return [
            SciSciNetDisruptionEnvCookbook(
                record=self.record,
                renderer=self.renderer,
                system_prompt=self.system_prompt,
                max_tokens=self.max_tokens,
                prompt_max_chars=self.prompt_max_chars,
                include_concepts=self.include_concepts,
            )
            for _ in range(int(self.group_size))
        ]

    async def compute_group_rewards(
        self, trajectory_group: list[Trajectory], env_group: Sequence[CookbookEnv]
    ) -> list[tuple[float, dict[str, float | int]]]:
        _ = trajectory_group, env_group
        return [(0.0, {}) for _ in range(int(self.group_size))]

    def logging_tags(self) -> list[str]:
        return ["sciscinet", "disruption", "single_turn"]

class SciSciNetCookbookRLDataset(CookbookRLDataset):
    def __init__(
        self,
        *,
        train_records: Sequence[Mapping[str, Any]],
        renderer: Any,
        groups_per_batch: int,
        group_size: int,
        n_batches: int,
        system_prompt: str | None,
        max_tokens: int,
        prompt_max_chars: int,
        include_concepts: bool = False,
        sampling_strategy: str = "natural",
        seed: int = 0,
    ) -> None:
        self.train_records = [dict(r) for r in train_records]
        self.renderer = renderer
        self.groups_per_batch = int(groups_per_batch)
        self.group_size = int(group_size)
        self.n_batches = int(n_batches)
        self.system_prompt = system_prompt
        self.max_tokens = int(max_tokens)
        self.prompt_max_chars = int(prompt_max_chars)
        self.include_concepts = bool(include_concepts)
        self.sampling_strategy = str(sampling_strategy).strip().lower()
        self.seed = int(seed)
        if not self.train_records:
            raise ValueError("train_records is empty")
        if self.groups_per_batch <= 0 or self.group_size <= 0 or self.n_batches <= 0:
            raise ValueError("groups_per_batch, group_size, and n_batches must be positive")
        if self.sampling_strategy not in {"natural", "stratified"}:
            raise ValueError("sampling_strategy must be 'natural' or 'stratified'")
        self._natural_records = [dict(r) for r in self.train_records]
        self._rng = random.Random(self.seed)
        self._rng.shuffle(self._natural_records)
        self._natural_pos = 0
        self._label_buckets: dict[str, list[dict[str, Any]]] = {label: [] for label in DISRUPTION_LABELS_PURE}
        for record in self.train_records:
            label = _normalize_label(record.get("disruption_label"))
            if label in self._label_buckets:
                self._label_buckets[label].append(dict(record))
        self._bucket_pos = {label: 0 for label in DISRUPTION_LABELS_PURE}
        for label, bucket in self._label_buckets.items():
            if bucket:
                self._rng.shuffle(bucket)
        if self.sampling_strategy == "stratified":
            empty = [label for label, bucket in self._label_buckets.items() if not bucket]
            if empty:
                raise ValueError(f"Cannot stratify: missing labels in train subset: {empty}")

    def __len__(self) -> int:
        return self.n_batches

    def _next_natural(self) -> dict[str, Any]:
        if self._natural_pos >= len(self._natural_records):
            self._natural_pos = 0
            self._rng.shuffle(self._natural_records)
        record = dict(self._natural_records[self._natural_pos])
        self._natural_pos += 1
        return record

    def _next_for_label(self, label: str) -> dict[str, Any]:
        bucket = self._label_buckets[label]
        pos = self._bucket_pos[label]
        if pos >= len(bucket):
            self._rng.shuffle(bucket)
            pos = 0
        self._bucket_pos[label] = pos + 1
        return dict(bucket[pos])

    def get_batch(self, index: int) -> Sequence[CookbookEnvGroupBuilder]:
        _ = index
        builders: list[SciSciNetEnvGroupBuilderCookbook] = []
        for item_idx in range(self.groups_per_batch):
            if self.sampling_strategy == "stratified":
                label = DISRUPTION_LABELS_PURE[item_idx % len(DISRUPTION_LABELS_PURE)]
                record = self._next_for_label(label)
            else:
                record = self._next_natural()
            builders.append(
                SciSciNetEnvGroupBuilderCookbook(
                    record=record,
                    group_size=self.group_size,
                    renderer=self.renderer,
                    system_prompt=self.system_prompt,
                    max_tokens=self.max_tokens,
                    prompt_max_chars=self.prompt_max_chars,
                    include_concepts=self.include_concepts,
                )
            )
        return builders

@dataclass
class SciSciNetCookbookRLDatasetBuilder:
    dataset_jsonl: str
    splits_json: str
    model_name_for_tokenizer: str
    renderer_name: str
    groups_per_batch: int
    group_size: int
    n_batches: int
    max_tokens: int
    prompt_max_chars: int
    seed: int = 0
    train_split: str = "train"
    val_split: str = "val"
    max_train: int | None = None
    max_val: int | None = None
    sampling_strategy: str = "natural"
    system_prompt: str | None = None
    include_concepts: bool = False

    async def __call__(self) -> tuple[SciSciNetCookbookRLDataset, None]:
        tokenizer = get_tokenizer(self.model_name_for_tokenizer)
        renderer = renderers.get_renderer(self.renderer_name, tokenizer=tokenizer)
        train_records, _val_records, _info = load_split_records_streaming_head_pure(
            dataset_jsonl=Path(self.dataset_jsonl),
            splits_json=Path(self.splits_json),
            train_split=self.train_split,
            val_split=self.val_split,
            max_train=self.max_train,
            max_val=self.max_val,
        )
        dataset = SciSciNetCookbookRLDataset(
            train_records=train_records,
            renderer=renderer,
            groups_per_batch=self.groups_per_batch,
            group_size=self.group_size,
            n_batches=self.n_batches,
            system_prompt=self.system_prompt,
            max_tokens=self.max_tokens,
            prompt_max_chars=self.prompt_max_chars,
            include_concepts=self.include_concepts,
            sampling_strategy=self.sampling_strategy,
            seed=self.seed,
        )
        return dataset, None


## Configure A Pure Cookbook RL Training Run

This mirrors the Cookbook `rl_basic` pattern (build `cookbook_rl_train.Config`) with your SciSciNet dataset files and a notebook-local dataset builder.
The config below is intentionally tiny (`n_batches=2`) so you can validate end-to-end before scaling up.


In [3]:
PURE_TINKER_TRAIN_CFG = {
    "model_name": "Qwen/Qwen3-8B",
    "lora_rank": 32,
    "learning_rate": 1e-5,
    "groups_per_batch": 6,
    "group_size": 4,
    "n_batches": 2,
    "max_tokens": 128,
    "prompt_max_chars": 1800,
    "max_train": 96,
    "max_val": 12,
    "sampling_strategy": "stratified",
    "include_concepts": False,
    "train_split": "train",
    "val_split": "val",
    "seed": SEED,
    "eval_every": 0,
    "save_every": 0,
    "temperature": 1.0,
    "base_url": os.environ.get("TINKER_BASE_URL"),
    "system_prompt": (
        "You are a scientific impact prediction model. Predict the disruption label and provide brief reasoning. "
        "Follow the output format exactly."
    ),
}

renderer_name_pure = model_info.get_recommended_renderer_name(PURE_TINKER_TRAIN_CFG["model_name"])
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pure_log_dir = TINKER_ROOT / "results" / "tinker_rl_cookbook" / f"notebook_pure_sciscinet_{timestamp}"

train_records_preview_pure, val_records_preview_pure, pure_data_info = load_split_records_streaming_head_pure(
    dataset_jsonl=DATASET_JSONL,
    splits_json=SPLITS_JSON,
    train_split=PURE_TINKER_TRAIN_CFG["train_split"],
    val_split=PURE_TINKER_TRAIN_CFG["val_split"],
    max_train=PURE_TINKER_TRAIN_CFG["max_train"],
    max_val=PURE_TINKER_TRAIN_CFG["max_val"],
)

pure_prompt_preview = build_pure_disruption_prompt(
    train_records_preview_pure[0],
    prompt_max_chars=PURE_TINKER_TRAIN_CFG["prompt_max_chars"],
    include_concepts=PURE_TINKER_TRAIN_CFG["include_concepts"],
) if train_records_preview_pure else ""

pure_dataset_builder = SciSciNetCookbookRLDatasetBuilder(
    dataset_jsonl=str(DATASET_JSONL),
    splits_json=str(SPLITS_JSON),
    model_name_for_tokenizer=PURE_TINKER_TRAIN_CFG["model_name"],
    renderer_name=renderer_name_pure,
    groups_per_batch=PURE_TINKER_TRAIN_CFG["groups_per_batch"],
    group_size=PURE_TINKER_TRAIN_CFG["group_size"],
    n_batches=PURE_TINKER_TRAIN_CFG["n_batches"],
    max_tokens=PURE_TINKER_TRAIN_CFG["max_tokens"],
    prompt_max_chars=PURE_TINKER_TRAIN_CFG["prompt_max_chars"],
    seed=PURE_TINKER_TRAIN_CFG["seed"],
    train_split=PURE_TINKER_TRAIN_CFG["train_split"],
    val_split=PURE_TINKER_TRAIN_CFG["val_split"],
    max_train=PURE_TINKER_TRAIN_CFG["max_train"],
    max_val=PURE_TINKER_TRAIN_CFG["max_val"],
    sampling_strategy=PURE_TINKER_TRAIN_CFG["sampling_strategy"],
    system_prompt=PURE_TINKER_TRAIN_CFG["system_prompt"],
    include_concepts=PURE_TINKER_TRAIN_CFG["include_concepts"],
)

cookbook_train_cfg = cookbook_rl_train.Config(
    learning_rate=PURE_TINKER_TRAIN_CFG["learning_rate"],
    dataset_builder=pure_dataset_builder,
    model_name=PURE_TINKER_TRAIN_CFG["model_name"],
    lora_rank=PURE_TINKER_TRAIN_CFG["lora_rank"],
    max_tokens=PURE_TINKER_TRAIN_CFG["max_tokens"],
    temperature=PURE_TINKER_TRAIN_CFG["temperature"],
    log_path=str(pure_log_dir),
    eval_every=PURE_TINKER_TRAIN_CFG["eval_every"],
    save_every=PURE_TINKER_TRAIN_CFG["save_every"],
    base_url=PURE_TINKER_TRAIN_CFG["base_url"],
    num_groups_to_log=0,
)

{
    "renderer_name": renderer_name_pure,
    "pure_data_info": pure_data_info,
    "log_path": str(pure_log_dir),
    "prompt_preview_chars": len(pure_prompt_preview),
    "prompt_preview_head": pure_prompt_preview[:700],
}


{'renderer_name': 'qwen3',
 'pure_data_info': {'dataset_jsonl': '/Users/akhilpandey/Downloads/disruption_novelty_sciscinet_500k.jsonl',
  'splits_json': '/Users/akhilpandey/Downloads/disruption_novelty_sciscinet_500k.splits.json',
  'train_split': 'train',
  'val_split': 'val',
  'requested_train_ids': 96,
  'requested_val_ids': 12,
  'loaded_train_records': 96,
  'loaded_val_records': 12,
  'train_label_histogram': {'disruptive': 0,
   'consolidating': 0,
   'neutral': 96},
  'val_label_histogram': {'disruptive': 0, 'consolidating': 0, 'neutral': 12}},
 'log_path': '/Users/akhilpandey/code/writing/tinker/results/tinker_rl_cookbook/notebook_pure_sciscinet_20260224_164933',
 'prompt_preview_chars': 2023,
 'prompt_preview_head': 'Predict the disruption label for the paper.\nAllowed labels: disruptive, consolidating, neutral.\nReturn exactly:\ndisruption: <label>\nreasoning: <short justification>\n\nTitle: Mapping almond stem water potential using machine learning and multispectral imager

In [4]:
RUN_PURE_TINKER_HOSTED_TRAIN = False
CLEAN_PURE_TINKER_LOGDIR_BEFORE_RUN = False

print("Training log dir:", cookbook_train_cfg.log_path)
if not os.environ.get("TINKER_API_KEY"):
    print("TINKER_API_KEY is not set in this notebook kernel. Set it before enabling hosted training.")

if RUN_PURE_TINKER_HOSTED_TRAIN:
    if CLEAN_PURE_TINKER_LOGDIR_BEFORE_RUN and Path(cookbook_train_cfg.log_path).exists():
        shutil.rmtree(cookbook_train_cfg.log_path)
    Path(cookbook_train_cfg.log_path).parent.mkdir(parents=True, exist_ok=True)
    # This is the actual hosted RL run (Cookbook rl_basic-style config + notebook-local env/dataset builder).
    await cookbook_rl_train.main(cookbook_train_cfg)

Training log dir: /Users/akhilpandey/code/writing/tinker/results/tinker_rl_cookbook/notebook_pure_sciscinet_20260224_164933


## Next Steps

- Flip `RUN_TRAIN_DRY_RUN` to `True` to validate the existing repository-helper smoke path end-to-end.
- For the pure Cookbook path, run the notebook under `/Users/akhilpandey/mlx/bin/python`, set `TINKER_API_KEY`, then flip `RUN_PURE_TINKER_HOSTED_TRAIN` to `True`.
- Start with `n_batches=2` and small `groups_per_batch`, then scale `max_train`, `n_batches`, and `max_tokens` after the first successful hosted run.
- If the pure notebook path works, port the notebook-local `Env` / `EnvGroupBuilder` / `RLDatasetBuilder` into repository modules and replace the current dry-run-only trainer branch.
